In [1]:
# Disable GPU before importing TensorFlow
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Check Python version
import sys
print('Python version:', sys.version)

# Some important functions and libraries
import numpy as np
import pandas as pd
from pathlib import Path
import random
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, average_precision_score

# Now import TensorFlow
import tensorflow as tf
print('Tensorflow version:', tf.__version__)
print("GPUs detected:", tf.config.list_physical_devices('GPU'))

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Tensorflow version: 2.20.0
GPUs detected: []


# Setup 

In [2]:
import os

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    candidate_roots = [
        Path('/content/drive/MyDrive/Customer-Appetency-Prediction-Kaggle-'),
        Path('/content/drive/MyDrive')
    ]
else:
    candidate_roots = [Path.cwd(), Path.cwd().parent]

PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / 'data' / 'processed').exists()),
    Path.cwd()
)
os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())

Mounted at /content/drive
Working directory: /content/drive/MyDrive


## Data loading and feature definition



In [3]:
train = pd.read_csv('data/processed/train_preprocessed_v2.csv', index_col=0)
test = pd.read_csv('data/processed/test_preprocessed_v2.csv', index_col=0)
train_labels = pd.read_csv('data/processed/y_train.csv', index_col=0)

feature_cols = [col for col in train.columns if col != 'ID']
X = train[feature_cols].copy()
X_submission = test[feature_cols].copy()
y = train_labels.iloc[:, 0].astype('float32')

print('Train shape:', train.shape)
print('Feature matrix shape:', X.shape)
print('Positive rate:', y.mean())

Train shape: (30000, 9279)
Feature matrix shape: (30000, 9279)
Positive rate: 0.0178


Quick inspection of the inputs



In [4]:
train.head(3)

,Var2,Var3,Var5,Var6,Var7,Var8,Var9,Var10,Var11,Var13,...,Var14992_is_missing,Var14994_is_missing,Var14995_is_missing,Var14996_is_missing,Var14997_is_missing,Var14998_is_missing,Var14999_is_missing,Var15000_is_missing,row_missing_count,row_missing_pct
ID,,,,,,,,,,,,,,,,,,,,,
502504,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,1,0,0,1,1,1,1,458,3.053333
197332,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,1,0,0,1,1,1,1,527,3.513333
277830,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,1,0,0,1,0,1,1,460,3.066667


In [5]:
train_labels.head(3)

,Target_appetency
ID,
502504,0
197332,0
277830,0


In [6]:
test.head(3)

,Var2,Var3,Var5,Var6,Var7,Var8,Var9,Var10,Var11,Var13,...,Var14992_is_missing,Var14994_is_missing,Var14995_is_missing,Var14996_is_missing,Var14997_is_missing,Var14998_is_missing,Var14999_is_missing,Var15000_is_missing,row_missing_count,row_missing_pct
ID,,,,,,,,,,,,,,,,,,,,,
877149,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,1,0,0,1,0,1,1,473,3.153333
109603,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,1,0,0,1,1,1,1,501,3.340000
560424,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,1,0,0,1,1,1,1,504,3.360000


# Baseline neural network

The first model is a straightforward multilayer used as a baseline. The objective is not only to obtain a score, but also to establish a reference architecture that can later be improved through stronger regularization and a smaller capacity.

## Train-validation split

The dataset is split into train and validation partitions with stratification. This is important because the target is highly imbalanced, so both splits must preserve approximately the same positive-class proportion. Using a fixed random seed makes the comparison between model 1 and model 2 reproducible.

In [7]:
#Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print('Train split:', X_train.shape)
print('Validation split:', X_test.shape)

Train split: (24000, 9279)
Validation split: (6000, 9279)


## Model 1 architecture

Model 1 uses batch normalization, two dense hidden layers, and dropout. This architecture gives the network enough flexibility to learn non-linear relationships in the processed tabular data while keeping the design simple enough to serve as a baseline for comparison.



The baseline is optimized with Adam and binary cross-entropy because the task is binary classification. Instead of tracking accuracy, the notebook focuses on ROC AUC and PR AUC, which are more informative under strong class imbalance.

In [8]:
model = tf.keras.models.Sequential([
    #Input layer
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.BatchNormalization(),
    #Layer 1
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    #Layer 2
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    #Output layer
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ batch_normalization             │ (None, 9279)           │        37,116 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,187,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,233,277 (4.70 MB)

 Trainable params: 1,214,719 (4.63 MB)

 Non-trainable params: 18,558 (72.49 KB)

In [ ]:
# Return prediction
predictions = model(X_train[:1]).numpy()
predictions

array([[1.]], dtype=float32)

In [ ]:
# loss function
loss_fn = tf.keras.losses.BinaryCrossentropy()
loss_fn(y_train.iloc[:1], predictions).numpy()

np.float32(15.942385)

In [ ]:
# Select the optimize
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.AUC(name='roc_auc', curve='ROC'),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR')
    ]
)

In [14]:
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train.to_numpy().ravel()
)

class_weight = dict(zip(classes, weights))

print(class_weight)

{np.float32(0.0): np.float64(0.5090569719594451), np.float32(1.0): np.float64(28.10304449648712)}


Baseline training strategy

Training uses early stopping monitored on validation PR AUC. This choice is because some improvements in precision recall behavior are more relevant than small changes in loss or accuracy and restoring the best weights ensures that the saved baseline corresponds to the best validation point reached during training.

In [15]:
# Train the model
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_pr_auc',
    mode='max',
    patience=8,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=128,
    callbacks=[early_stop],
    class_weight=class_weight
)

Epoch 1/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 1.1920 - pr_auc: 0.0270 - roc_auc: 0.6065 - val_loss: 0.9239 - val_pr_auc: 0.0384 - val_roc_auc: 0.6942
Epoch 2/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.6245 - pr_auc: 0.1125 - roc_auc: 0.8285 - val_loss: 0.4594 - val_pr_auc: 0.0388 - val_roc_auc: 0.6493
Epoch 3/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3189 - pr_auc: 0.3771 - roc_auc: 0.9425 - val_loss: 0.3799 - val_pr_auc: 0.0509 - val_roc_auc: 0.6210
Epoch 4/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2011 - pr_auc: 0.6127 - roc_auc: 0.9751 - val_loss: 0.3724 - val_pr_auc: 0.0512 - val_roc_auc: 0.6391
Epoch 5/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1378 - pr_auc: 0.6949 - roc_auc: 0.9886 - val_loss: 0.3134 - val_pr_auc: 0.0422 - val_roc_auc: 0.6018
Epoch 6/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1379 - pr_auc: 0.6917 - roc_auc: 0.9865 - val_loss: 0.3045 - val_pr_auc: 0.0507 - val_roc_auc: 0.6462
Epoch 7/50
188/1

In [16]:
# Evaluate the model with the loss function and the accuracy
model.evaluate(X_train,  y_train, verbose=2)

750/750 - 0s - 656us/step - loss: 0.1258 - pr_auc: 0.8628 - roc_auc: 0.9985


[0.12584374845027924, 0.9984743595123291, 0.8627500534057617]

In [17]:
# Evaluate the model with the loss function and the accuracy
model.evaluate(X_test,  y_test, verbose=2)

188/188 - 0s - 769us/step - loss: 0.3040 - pr_auc: 0.0574 - roc_auc: 0.6389


[0.30400896072387695, 0.6388627290725708, 0.0574495829641819]



The baseline shows a large gap between training and validation performance. This suggests overfitting: the network captures the training set very well but is not able to generalize

In [18]:
# The model already returns probabilities
probability_model = model

In [19]:
# Make prediction
y_test_pred = probability_model.predict(X_test, verbose=0)
y_test_pred[:5]

array([[0.00020847],
       [0.05233803],
       [0.12396932],
       [0.08071485],
       [0.00313337]], dtype=float32)

## Model 2: regularized alternative

The second model is designed as an explicit improvement attempt over model 1. The main changes are:

1. The hidden representation is reduced from two layers to a single layer with 32 neurons, which lowers model capacity and reduces the risk of memorizing noise.
2. A stronger dropout rate (0.4) is used 
3. L1/L2 regularization is added 
4. The learning rate is reduced and the batch size is increased

## Comparison between model 1 and model 2

Both models are evaluated on the same validation split. The comparison table summarizes the best validation PR AUC, the best validation ROC AUC, and the minimum validation loss reached during training. In this notebook, PR AUC is the main selection criterion because it better reflects the quality of ranking the rare positive class.

Note: concepts like batchnormalization, and dense layers where seen in deep learning class

In [20]:
alt_model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(
        32,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l1_l2(l1=1e-6, l2=1e-4)
    ),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

alt_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.AUC(name='roc_auc', curve='ROC'),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR')
    ]
)

alt_early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_pr_auc',
    mode='max',
    patience=8,
    restore_best_weights=True
)

alt_history = alt_model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=256,
    callbacks=[alt_early_stop],
    class_weight=class_weight,
    verbose=2
)

baseline_eval = dict(zip(model.metrics_names, model.evaluate(X_test, y_test, verbose=0)))
alt_eval = dict(zip(alt_model.metrics_names, alt_model.evaluate(X_test, y_test, verbose=0)))

comparison = pd.DataFrame(
    [
        {
            'best_val_pr_auc': max(history.history['val_pr_auc']),
            'best_val_roc_auc': max(history.history['val_roc_auc']),
            'best_val_loss': min(history.history['val_loss']),
        },
        {
            'best_val_pr_auc': max(alt_history.history['val_pr_auc']),
            'best_val_roc_auc': max(alt_history.history['val_roc_auc']),
            'best_val_loss': min(alt_history.history['val_loss']),
        }
    ],
    index=['baseline_mlp', 'regularized_shallow_mlp']
).sort_values('best_val_pr_auc', ascending=False)

comparison

Epoch 1/50
94/94 - 1s - 14ms/step - loss: 1.1414 - pr_auc: 0.0278 - roc_auc: 0.5960 - val_loss: 1.3548 - val_pr_auc: 0.0251 - val_roc_auc: 0.5966
Epoch 2/50
94/94 - 0s - 4ms/step - loss: 0.5577 - pr_auc: 0.2292 - roc_auc: 0.8609 - val_loss: 0.6569 - val_pr_auc: 0.0404 - val_roc_auc: 0.6342
Epoch 3/50
94/94 - 0s - 4ms/step - loss: 0.3640 - pr_auc: 0.5388 - roc_auc: 0.9413 - val_loss: 0.5105 - val_pr_auc: 0.0475 - val_roc_auc: 0.6312
Epoch 4/50
94/94 - 0s - 4ms/step - loss: 0.3173 - pr_auc: 0.5340 - roc_auc: 0.9535 - val_loss: 0.4282 - val_pr_auc: 0.0565 - val_roc_auc: 0.6274
Epoch 5/50
94/94 - 0s - 4ms/step - loss: 0.2416 - pr_auc: 0.7233 - roc_auc: 0.9769 - val_loss: 0.3600 - val_pr_auc: 0.0544 - val_roc_auc: 0.6209
Epoch 6/50
94/94 - 0s - 4ms/step - loss: 0.1951 - pr_auc: 0.8393 - roc_auc: 0.9859 - val_loss: 0.2840 - val_pr_auc: 0.0768 - val_roc_auc: 0.6184
Epoch 7/50
94/94 - 0s - 4ms/step - loss: 0.1807 - pr_auc: 0.8134 - roc_auc: 0.9862 - val_loss: 0.3252 - val_pr_auc: 0.0540 - val_

,best_val_pr_auc,best_val_roc_auc,best_val_loss
regularized_shallow_mlp,0.087467,0.634153,0.236529
baseline_mlp,0.057450,0.694206,0.283090


![nn_image](NN_IMAGE.png)


Model 2 improves validation PR AUC and lowers validation loss, which indicates that the smaller and more regularized architecture generalizes better than model 1. The improvement is meaningful, but the validation scores remain low, so the overfitting issue is reduced rather than completely removed.

In [ ]:
submission_nn = pd.DataFrame({
    'ID': test.index,
    'Target_appetency': model.predict(X_submission, verbose=0).ravel()
})

submission_nn.to_csv('submission_nn_proba.csv', index=False)
submission_nn.head()